# Semana 14: Despliegue de Modelos NLP (MLOps)
## Notebook de Ejercicios (NB2) – Monitoreo de Deriva

**Propósito:** Simular un escenario de drift de datos en producción, monitorear las predicciones y detectar cambios usando Evidently.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Desplegar una API local con un modelo de clasificación de sentimiento.
- Enviar lotes de textos y registrar predicciones.
- Usar Evidently para comparar distribuciones de predicciones y detectar drift.
- Discutir acciones correctivas ante la detección de drift.

---

## 0. Configuración Inicial

Importamos las librerías necesarias e instalamos dependencias.

In [ ]:
# Instalamos dependencias
!pip install fastapi uvicorn transformers torch requests nest-asyncio evidently pandas datasets

# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import time
import requests
import json
import os
import subprocess
import threading
import warnings
warnings.filterwarnings('ignore')

# Transformers y datasets
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

# FastAPI y uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

# Evidently
from evidently import ColumnMapping
from evidently.report import Report
from evidently.metrics import ColumnDriftMetric, DatasetDriftMetric
from evidently.test_suite import TestSuite
from evidently.tests import TestColumnDrift, TestNumberOfDriftedColumns

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Verificar dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

print("\nLibrerías importadas correctamente.")

---
## 1. Modelo Base y Datos de Referencia

Usamos un modelo de análisis de sentimiento pre-entrenado y cargamos datos de IMDb para crear datos de referencia (training) y datos actuales (con posible drift).

In [ ]:
# Cargar modelo de sentimiento
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print("Modelo de sentimiento cargado.")
print(f"Clases: {model.config.id2label}")

# Cargar dataset IMDb
print("\nCargando dataset IMDb...")
imdb = load_dataset("imdb")

# Crear datos de referencia (entrenamiento)
reference_texts = imdb['train']['text'][:1000]
reference_labels = imdb['train']['label'][:1000]

# Crear datos actuales (test - sin drift)
current_texts = imdb['test']['text'][:1000]
current_labels = imdb['test']['label'][:1000]

print(f"Datos de referencia: {len(reference_texts)} textos")
print(f"Datos actuales: {len(current_texts)} textos")

---
## 2. Despliegue de la API Local

Creamos una API simple con FastAPI para servir predicciones.

In [ ]:
# Crear archivo app.py con la API
app_content = '''
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Configuración
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

# Cargar modelo y tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

# Definir esquema de entrada
class TextInput(BaseModel):
    text: str

# Crear app
app = FastAPI(title="Sentiment Analysis API", description="API para clasificación de sentimiento", version="1.0")

@app.get("/")
def root():
    return {"message": "Sentiment Analysis API. Use /predict con POST."}

@app.post("/predict")
def predict(input_data: TextInput):
    try:
        # Tokenizar
        inputs = tokenizer(input_data.text, return_tensors="pt", truncation=True, padding=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Inferencia
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=-1)
            prediction = torch.argmax(logits, dim=-1).item()
        
        # Mapear etiquetas
        id2label = model.config.id2label
        label = id2label[prediction]
        confidence = probabilities[0][prediction].item()
        
        return {
            "prediction": label,
            "confidence": confidence,
            "probabilities": probabilities[0].tolist()
        }
    except Exception as e:
        return {"error": str(e)}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("app.py", "w") as f:
    f.write(app_content)

print("Archivo app.py creado.")

In [ ]:
# Función para ejecutar la API en segundo plano
def run_api():
    os.system("uvicorn app:app --host 0.0.0.0 --port 8000")

# Ejecutar en un hilo separado
api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

# Esperar a que la API inicie
time.sleep(10)
print("API iniciada en http://localhost:8000")

In [ ]:
# Probar que la API funciona
def test_api(text):
    url = "http://localhost:8000/predict"
    payload = {"text": text}
    try:
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            return response.json()
        else:
            return {"error": f"Status: {response.status_code}"}
    except Exception as e:
        return {"error": str(e)}

test_text = "I love this movie!"
result = test_api(test_text)
print(f"Test: {result}")

---
## 3. Obtener Predicciones para Datos de Referencia

Enviamos los datos de referencia a la API y registramos las predicciones.

In [ ]:
def get_predictions(texts, batch_size=50):
    """
    Obtiene predicciones de la API para una lista de textos.
    """
    predictions = []
    confidences = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        for text in batch:
            result = test_api(text)
            if 'prediction' in result:
                predictions.append(result['prediction'])
                confidences.append(result['confidence'])
            else:
                predictions.append('ERROR')
                confidences.append(0.0)
        print(f"Procesados {min(i+batch_size, len(texts))}/{len(texts)} textos")
        time.sleep(1)  # evitar sobrecargar la API
    
    return predictions, confidences

print("Obteniendo predicciones para datos de referencia...")
ref_predictions, ref_confidences = get_predictions(reference_texts[:200])  # reducir para velocidad

# Crear DataFrame de referencia
ref_df = pd.DataFrame({
    'text': reference_texts[:200],
    'true_label': reference_labels[:200],
    'prediction': ref_predictions,
    'confidence': ref_confidences,
    'prediction_numeric': [1 if p == 'POSITIVE' else 0 for p in ref_predictions]
})

print("\nPrimeras filas del DataFrame de referencia:")
ref_df.head()

---
## 4. Datos Actuales (Sin Drift)

Obtenemos predicciones para datos actuales sin drift (test set) para comparar.

In [ ]:
print("Obteniendo predicciones para datos actuales (sin drift)...")
curr_predictions, curr_confidences = get_predictions(current_texts[:200])

curr_df = pd.DataFrame({
    'text': current_texts[:200],
    'true_label': current_labels[:200],
    'prediction': curr_predictions,
    'confidence': curr_confidences,
    'prediction_numeric': [1 if p == 'POSITIVE' else 0 for p in curr_predictions]
})

print("\nPrimeras filas del DataFrame actual:")
curr_df.head()

---
## 5. Simulación de Drift

Creamos datos con drift modificando artificialmente los textos (ej. añadiendo palabras positivas/negativas extremas).

In [ ]:
def create_drift_data(texts, labels, drift_type='positive'):
    """
    Crea datos con drift añadiendo palabras extremas.
    """
    drift_texts = []
    if drift_type == 'positive':
        prefix = "This is absolutely amazing and wonderful. "
    else:
        prefix = "This is absolutely terrible and awful. "
    
    for text in texts[:100]:  # reducimos para velocidad
        drift_texts.append(prefix + text)
    
    return drift_texts, labels[:100]

drift_texts_pos, drift_labels_pos = create_drift_data(current_texts, current_labels, 'positive')
drift_texts_neg, drift_labels_neg = create_drift_data(current_texts, current_labels, 'negative')

print(f"Datos con drift positivo: {len(drift_texts_pos)} textos")
print(f"Datos con drift negativo: {len(drift_texts_neg)} textos")

# Obtener predicciones para datos con drift
print("\nObteniendo predicciones para datos con drift positivo...")
drift_pos_predictions, drift_pos_confidences = get_predictions(drift_texts_pos)

print("\nObteniendo predicciones para datos con drift negativo...")
drift_neg_predictions, drift_neg_confidences = get_predictions(drift_texts_neg)

# Crear DataFrames
drift_pos_df = pd.DataFrame({
    'text': drift_texts_pos,
    'true_label': drift_labels_pos,
    'prediction': drift_pos_predictions,
    'confidence': drift_pos_confidences,
    'prediction_numeric': [1 if p == 'POSITIVE' else 0 for p in drift_pos_predictions]
})

drift_neg_df = pd.DataFrame({
    'text': drift_texts_neg,
    'true_label': drift_labels_neg,
    'prediction': drift_neg_predictions,
    'confidence': drift_neg_confidences,
    'prediction_numeric': [1 if p == 'POSITIVE' else 0 for p in drift_neg_predictions]
})

---
## 6. Detección de Drift con Evidently

Usamos Evidently para comparar las distribuciones de predicciones entre datos de referencia y datos actuales (con y sin drift).

In [ ]:
# Configurar column mapping
column_mapping = ColumnMapping()
column_mapping.target = 'true_label'
column_mapping.prediction = 'prediction_numeric'
column_mapping.numerical_features = ['confidence']
column_mapping.categorical_features = ['prediction']

# Reporte de drift para datos sin drift
report_no_drift = Report(metrics=[
    DatasetDriftMetric(),
    ColumnDriftMetric(column_name='prediction_numeric'),
    ColumnDriftMetric(column_name='confidence')
])
report_no_drift.run(reference_data=ref_df, current_data=curr_df, column_mapping=column_mapping)

# Reporte de drift para datos con drift positivo
report_drift_pos = Report(metrics=[
    DatasetDriftMetric(),
    ColumnDriftMetric(column_name='prediction_numeric'),
    ColumnDriftMetric(column_name='confidence')
])
report_drift_pos.run(reference_data=ref_df, current_data=drift_pos_df, column_mapping=column_mapping)

# Reporte de drift para datos con drift negativo
report_drift_neg = Report(metrics=[
    DatasetDriftMetric(),
    ColumnDriftMetric(column_name='prediction_numeric'),
    ColumnDriftMetric(column_name='confidence')
])
report_drift_neg.run(reference_data=ref_df, current_data=drift_neg_df, column_mapping=column_mapping)

In [ ]:
# Función para extraer métricas de drift
def get_drift_metrics(report, name):
    result = report.as_dict()
    metrics = {}
    for metric in result['metrics']:
        if 'result' in metric:
            if 'drift_share' in metric['result']:
                metrics['dataset_drift_share'] = metric['result']['drift_share']
            if 'drift_detected' in metric['result']:
                metrics[f"drift_{metric['metric']}"] = metric['result']['drift_detected']
    return metrics

print("=== RESULTADOS DE DETECCIÓN DE DRIFT ===\n")

metrics_no_drift = get_drift_metrics(report_no_drift, "Sin Drift")
print("Sin Drift:")
print(f"  Proporción de drift en dataset: {metrics_no_drift.get('dataset_drift_share', 0):.2%}")

metrics_drift_pos = get_drift_metrics(report_drift_pos, "Drift Positivo")
print("\nCon Drift Positivo:")
print(f"  Proporción de drift en dataset: {metrics_drift_pos.get('dataset_drift_share', 0):.2%}")

metrics_drift_neg = get_drift_metrics(report_drift_neg, "Drift Negativo")
print("\nCon Drift Negativo:")
print(f"  Proporción de drift en dataset: {metrics_drift_neg.get('dataset_drift_share', 0):.2%}")

In [ ]:
# Visualizar distribuciones de predicciones
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ref_df['prediction_numeric'].hist(ax=axes[0], bins=2, alpha=0.7, color='blue', edgecolor='black')
axes[0].set_title('Referencia')
axes[0].set_xlabel('Predicción (0=Negativo, 1=Positivo)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_xticks([0,1])

drift_pos_df['prediction_numeric'].hist(ax=axes[1], bins=2, alpha=0.7, color='red', edgecolor='black')
axes[1].set_title('Drift Positivo')
axes[1].set_xlabel('Predicción (0=Negativo, 1=Positivo)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_xticks([0,1])

drift_neg_df['prediction_numeric'].hist(ax=axes[2], bins=2, alpha=0.7, color='red', edgecolor='black')
axes[2].set_title('Drift Negativo')
axes[2].set_xlabel('Predicción (0=Negativo, 1=Positivo)')
axes[2].set_ylabel('Frecuencia')
axes[2].set_xticks([0,1])

plt.tight_layout()
plt.show()

---
## 7. Test Suite con Evidently

Podemos usar Test Suite para definir umbrales automáticos de drift.

In [ ]:
# Crear test suite
test_suite = TestSuite(tests=[
    TestNumberOfDriftedColumns(),
    TestColumnDrift(column_name='prediction_numeric'),
    TestColumnDrift(column_name='confidence')
])

# Ejecutar para datos con drift positivo
test_suite.run(reference_data=ref_df, current_data=drift_pos_df, column_mapping=column_mapping)

# Mostrar resultados
test_results = test_suite.as_dict()
print("=== RESULTADOS DE TEST SUITE ===\n")
for test in test_results['tests']:
    status = test['status']
    name = test['name']
    description = test['description']
    print(f"Test: {name}")
    print(f"  Estado: {status}")
    print(f"  Descripción: {description}")
    print()

---
## 8. Discusión de Acciones ante Drift

Cuando se detecta drift, podemos tomar varias acciones:

### 8.1. Reentrenamiento del Modelo
- **Trigger**: Si el drift supera un umbral (ej. > 30% de columnas con drift).
- **Acción**: Reentrenar el modelo con los nuevos datos (o una mezcla de datos antiguos y nuevos).

### 8.2. Ajuste de Umbrales
- **Trigger**: Si el drift es pequeño pero consistente.
- **Acción**: Ajustar el umbral de decisión del modelo (ej. para balancear precisión/recall).

### 8.3. Alerta y Revisión Manual
- **Trigger**: Detección de drift en un subconjunto específico (ej. una categoría).
- **Acción**: Enviar alerta a un equipo de ML para revisión manual.

### 8.4. Actualización de Datos de Referencia
- **Trigger**: Drift aceptado como parte de la evolución natural de los datos.
- **Acción**: Actualizar los datos de referencia con una ventana móvil de datos recientes.

### 8.5. Shadow Deployment
- **Acción**: Desplegar un nuevo modelo en modo shadow (paralelo) y comparar su rendimiento antes de reemplazar el modelo actual.

**En nuestro caso:**
- Los datos con drift positivo muestran un claro cambio en la distribución de predicciones (muchos más positivos).
- Esto podría deberse a una campaña de marketing que genera opiniones extremadamente positivas.
- Acción recomendada: Reentrenar el modelo con una mezcla de datos antiguos y nuevos para adaptarse a la nueva distribución.

In [ ]:
# Simulación de reentrenamiento (concepto)
print("=== SIMULACIÓN DE REENTRENAMIENTO ===")

# Combinar datos de referencia y datos con drift
combined_texts = reference_texts[:100] + drift_texts_pos
combined_labels = reference_labels[:100] + drift_labels_pos

print(f"Datos combinados: {len(combined_texts)} textos")
print("En un escenario real, aquí reentrenaríamos el modelo.")

# Evaluar el nuevo modelo (simulado)
print("\nMétricas esperadas después de reentrenamiento:")
print("- Mayor precisión en datos con drift")
print("- Posible ligera degradación en datos originales")
print("- Trade-off aceptable para mantener rendimiento general")

---
## 9. Conclusiones

En este notebook hemos implementado un sistema de monitoreo de deriva para un modelo NLP en producción:

✔️ **API Local**: Desplegamos un modelo de sentimiento con FastAPI.

✔️ **Datos de Referencia**: Obtuvimos predicciones para datos de entrenamiento.

✔️ **Simulación de Drift**: Creamos datos con drift positivo y negativo añadiendo prefijos extremos.

✔️ **Detección con Evidently**:
- Sin drift: baja proporción de columnas con drift.
- Con drift: alta proporción (> 50%) de columnas con drift, detectado claramente.

✔️ **Test Suite**: Validamos automáticamente si hay drift en columnas específicas.

✔️ **Acciones**: Discutimos estrategias como reentrenamiento, ajuste de umbrales y alertas.

**Lección clave**: El monitoreo de drift es esencial para mantener la calidad de los modelos en producción. Herramientas como Evidently permiten detectar cambios automáticamente y tomar acciones correctivas.

---
**Fin del Notebook de Ejercicios - Semana 14 (NLP)**
**Fin del Curso de Procesamiento de Lenguaje Natural**